## Q1: Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application. 

- **Driver**: The process running `main()`/the SparkSession. It builds the DAG of
  transformations, splits it into stages and tasks, and schedules those tasks. It also
  collects results back (e.g. for `.collect()`, `.show()`) and holds the SparkContext.
- **Cluster Manager**: Negotiates and allocates resources (CPU/memory) across the cluster
  for the application. Examples: Standalone, YARN, Kubernetes, Mesos. It launches executor
  processes on worker nodes on the Driver's behalf but does not execute Spark tasks itself.
- **Executor**: A JVM process on a worker node that actually runs the tasks (transformations
  on partitions of data) assigned by the Driver, stores data in memory/disk for caching, and
  reports status/results back to the Driver. Each executor runs many tasks in parallel across
  its cores.

Flow: Driver → asks Cluster Manager for resources → Cluster Manager launches Executors → Driver sends tasks to Executors → Executors compute and return results/status to Driver.

--------------------

## Q2: How does Spark’s Lazy Evaluation strategy improve performance when chain-processing large datasets? 

Spark does not execute a transformation (`select`, `filter`, `withColumn`, `join`, etc.) the
moment it's called. It just records it as a step in a logical plan (lineage). Execution only
happens when an **action** is triggered (`show`, `count`, `write`, `collect`).

Why this helps with chained processing on large data:
1. **Query optimization** – Catalyst can look at the *entire* chain of transformations at once
   and reorder/merge operations (e.g. combine multiple `filter`s, push filters down before
   `select`s) before running anything.
2. **Predicate/Projection pushdown** – filters and column pruning can be pushed all the way
   down to the data source (e.g. Parquet), so unnecessary data is never even read.
3. **Avoids wasted work** – no intermediate DataFrame is ever materialized in full unless
   cached; only the data actually needed for the final action is computed.
4. **Pipelining** – narrow transformations get fused into a single stage that streams
   row-by-row through partitions, instead of writing intermediate results to disk between
   every step.

Net effect: on a long chain of `.filter().select().withColumn().groupBy()...`, Spark runs one
optimized physical plan instead of naively executing each step eagerly.

--------------------

## Q3: Write a Spark command to read a CSV file located at "data/source.csv", ensuring the first row is treated as a header and inferSchema is enabled. 


In [32]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("Week_06_Assignment").getOrCreate()

df = spark.read.csv("data/source.csv",header=True,inferSchema=True)

In [33]:
df.printSchema()      # shows column names and types
df.show(5)            # shows first 5 rows

root
 |-- product_id: string (nullable = true)
 |-- old_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- base_price: double (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)

+----------+--------+-------------+--------+----------+---------+--------+-------+------+--------+
|product_id|old_name|     category|   price|base_price|   status|  amount|user_id|region|priority|
+----------+--------+-------------+--------+----------+---------+--------+-------+------+--------+
|     P0001|   Table|    Furniture|75084.45|  61840.99|  Pending|75084.45|   1001|  West|     Low|
|     P0002|  Bottle|      Kitchen|15357.29|  13391.34|  Pending|15357.29|   1002|  East|    High|
|     P0003|   Phone|  Electronics|45308.08|  45152.67|Completed|45308.08|   1003| North|    High|
|    

--------------------

## Q4: What is the difference between CSV and Parquet in terms of storage (row-based vs. columnar) and why does it matter for performance? 


- **CSV** is **row-based**: each full row is stored together. To read even one column,
  Spark must read and parse *every* column of *every* row (line by line, text parsing).
  No compression benefit from repeated values across a column, no schema stored in the file
  (needs `inferSchema` scan or manual schema).
- **Parquet** is **columnar**: values of the same column are stored contiguously, per
  "row group", along with min/max statistics and the schema itself.

Why it matters for performance:
- **I/O reduction**: if a query only needs 2 of 20 columns, Parquet reads only those
  column chunks; CSV has to read the whole file.
- **Predicate pushdown**: Parquet's per-row-group statistics let Spark skip entire blocks
  of data that can't match a filter, without decompressing them (see Q9).
- **Compression**: columnar data (similar values together) compresses far better than
  row-based text, shrinking storage and disk I/O.
- **Schema on read**: Parquet embeds the schema, so no expensive `inferSchema` scan pass
  is needed (CSV inferSchema requires an extra full read of the data).
      

--------------------

## Q5: Given a DataFrame df, write a query to select the columns product_id and price where the category is 'Electronics'. 

In [3]:
result_Q5 = df.select("product_id", "price").filter(df.category == "Electronics")
result_Q5.show()

+----------+--------+
|product_id|   price|
+----------+--------+
|     P0003|45308.08|
|     P0008|94673.14|
|     P0010|81745.74|
|     P0011|23005.38|
|     P0012|25793.63|
|     P0016| 39899.9|
|     P0017|15996.76|
|     P0024| 4195.12|
|     P0025| 1904.71|
|     P0029| 5994.38|
|     P0035|23920.98|
|     P0036|77987.96|
|     P0037| 7565.65|
|     P0038|20804.27|
|     P0042|66736.03|
|     P0044|  1061.2|
|     P0054|48045.74|
|     P0056| 43365.5|
|     P0057|76589.57|
|     P0061|43131.34|
+----------+--------+
only showing top 20 rows


--------------------

## Q6: Write the code to "revise" a DataFrame by renaming the column old_name to new_name and casting the price column from a String to a Double. 

In [5]:
df = df.withColumnRenamed("old_name", "new_name") \
       .withColumn("price", col("price").cast("double"))

df.printSchema() 

df.show(5)

root
 |-- product_id: string (nullable = true)
 |-- new_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- base_price: double (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)

+----------+--------+-------------+--------+----------+---------+--------+-------+------+--------+
|product_id|new_name|     category|   price|base_price|   status|  amount|user_id|region|priority|
+----------+--------+-------------+--------+----------+---------+--------+-------+------+--------+
|     P0001|   Table|    Furniture|75084.45|  61840.99|  Pending|75084.45|   1001|  West|     Low|
|     P0002|  Bottle|      Kitchen|15357.29|  13391.34|  Pending|15357.29|   1002|  East|    High|
|     P0003|   Phone|  Electronics|45308.08|  45152.67|Completed|45308.08|   1003| North|    High|
|    

--------------------

## Q7: How does Spark use the Lineage Graph (DAG) to provide fault tolerance if a worker node fails? 

RDDs/DataFrames don't store data redundantly for safety — instead each one remembers its
**lineage**: the sequence of transformations (and the parent RDD/partition) that produced it,
forming a DAG.

If a worker node (and its executor) fails partway through a job:
1. The Driver notices the executor/task is lost (missed heartbeat).
2. Since the lost partition's *lineage* is known (it's just "transformation X applied to
   partition Y of source Z"), Spark **recomputes only that lost partition** from the original
   data / last checkpoint / cached parent — not the whole dataset.
3. The recomputed task is rescheduled on a healthy executor and the job continues.

This is why Spark doesn't need replicated in-memory copies of every intermediate result
(unlike traditional distributed systems that replicate data for durability) — the DAG itself
is the recovery mechanism. `checkpoint()` can also be used to truncate very long lineages by
persisting a materialized version to reliable storage.


--------------------

## Q8: Write a query to filter a DataFrame df_orders for rows where the status is 'Completed' AND the amount is greater than 1000. 

In [36]:
result_Q8 = df.filter((col("status") == "Completed") & (col("amount") > 1000))
result_Q8.show()

+----------+----------+-------------+--------+----------+---------+--------+-------+------+--------+
|product_id|  old_name|     category|   price|base_price|   status|  amount|user_id|region|priority|
+----------+----------+-------------+--------+----------+---------+--------+-------+------+--------+
|     P0003|     Phone|  Electronics|45308.08|  45152.67|Completed|45308.08|   1003| North|    High|
|     P0005|       Fan|HomeAppliance|70929.06|  66631.99|Completed|70929.06|   1005|  East|  Medium|
|     P0006|     Watch|  Accessories|16990.38|  15016.24|Completed|16990.38|   1006|  East|    High|
|     P0007|     Watch|  Accessories| 66935.5|  65899.55|Completed| 66935.5|   1007|  West|    High|
|     P0011|     Mouse|  Electronics|23005.38|  21123.57|Completed|23005.38|   1011| North|     Low|
|     P0013|     Shoes|     Clothing|19023.14|  18364.97|Completed|19023.14|   1013|  West|     Low|
|     P0014|     Mixer|HomeAppliance|57544.53|  50669.24|Completed|57544.53|   1014| South|

--------------------

## Q9: Explain the concept of Predicate Pushdown in Parquet and how it affects the amount of data loaded into memory. 

Predicate pushdown means the **filter condition itself is pushed down into the file-reading
layer**, rather than being applied after all data is loaded into Spark's memory.

Because Parquet stores column statistics (min/max, null counts) per row-group/column chunk:
- Spark's Catalyst optimizer sees a filter like `price > 1000` in the query plan.
- It passes that predicate down to the Parquet reader.
- The reader checks each row-group's stored min/max for that column **before decompressing
  or reading the actual column data**.
- Entire row-groups that can't possibly satisfy the predicate (e.g. max price in that group
  is 500) are **skipped entirely** — not decompressed, not deserialized, not loaded into JVM
  memory.

Effect on memory/IO: only the row-groups (and only the columns, combined with projection
pushdown) that could contain matching rows are actually read and materialized in memory.
For large datasets with selective filters, this can cut I/O and memory usage by orders of
magnitude compared to reading everything and filtering afterward — which is exactly what
happens with row-based formats like CSV that have no per-block statistics to skip on.

--------------------

## Q10: Write a code snippet to add a new column final_price which is the base_price multiplied by 1.18 (18% tax). 

In [16]:
df_tax = df.withColumn("final_price", (col("base_price") * 1.18))
df_tax.select("product_id", "base_price", "final_price").show(5)

+----------+----------+-----------------+
|product_id|base_price|      final_price|
+----------+----------+-----------------+
|     P0001|  61840.99|       72972.3682|
|     P0002|  13391.34|       15801.7812|
|     P0003|  45152.67|53280.15059999999|
|     P0004|  23301.41|       27495.6638|
|     P0005|  66631.99|       78625.7482|
+----------+----------+-----------------+
only showing top 5 rows


--------------------

## Q11: What is the difference between Transformations and Actions? Provide two examples of each. 

- **Transformations** are **lazy** — they define a new DataFrame/RDD from an existing one but
  don't execute anything immediately. They just get added to the lineage/DAG.
  - Examples: `filter()`, `select()`, `withColumn()`, `groupBy()`, `join()`, `map()`.
- **Actions** **trigger execution** of the accumulated DAG and return a result to the Driver
  or write output.
  - Examples: `show()`, `count()`, `collect()`, `write.parquet()`, `take()`, `first()`.

Rule of thumb: if it returns another DataFrame → transformation; if it returns a
value/output outside Spark (to the driver or to storage) → action.

--------------------

## Q12: Write the Spark command to load a Parquet file from "path/to/input", filter out any rows where user_id is null, and save the result as a CSV at "path/to/output". 

In [31]:
df.write.mode("overwrite").parquet("data/source_parquet")

orders_df = spark.read.parquet("data/source_parquet")

orders_df.filter(
    orders_df.user_id.isNotNull()
).write.mode("overwrite") \
 .option("header", True) \
 .csv("data/output")

+----------+----------+-------------+--------+----------+---------+--------+-------+------+--------+
|product_id|  new_name|     category|   price|base_price|   status|  amount|user_id|region|priority|
+----------+----------+-------------+--------+----------+---------+--------+-------+------+--------+
|     P0001|     Table|    Furniture|75084.45|  61840.99|  Pending|75084.45|   1001|  West|     Low|
|     P0002|    Bottle|      Kitchen|15357.29|  13391.34|  Pending|15357.29|   1002|  East|    High|
|     P0003|     Phone|  Electronics|45308.08|  45152.67|Completed|45308.08|   1003| North|    High|
|     P0005|       Fan|HomeAppliance|70929.06|  66631.99|Completed|70929.06|   1005|  East|  Medium|
|     P0006|     Watch|  Accessories|16990.38|  15016.24|Completed|16990.38|   1006|  East|    High|
|     P0007|     Watch|  Accessories| 66935.5|  65899.55|Completed| 66935.5|   1007|  West|    High|
|     P0008|Headphones|  Electronics|94673.14|  77579.14|Cancelled|94673.14|   1008| South|

--------------------

## Q13: In Spark Architecture, what is the difference between Client Mode and Cluster Mode? 

- **Client Mode**: The **Driver runs on the machine that submitted the job** (e.g. your
  laptop, an edge node, a notebook server) — outside the cluster. Only the Executors run on
  the cluster. Good for interactive work (spark-shell, notebooks) since you see driver
  logs/output directly, but if the client machine disconnects or dies, the whole application
  dies with it. Also means driver-executor communication crosses the network to the client.

- **Cluster Mode**: The **Driver itself is launched inside the cluster** (on one of the worker
  nodes), managed by the Cluster Manager, alongside the Executors. Better for production/batch
  jobs (`spark-submit --deploy-mode cluster`) — the job survives the submitting client
  disconnecting, and driver-executor communication stays inside the cluster network (lower
  latency). You don't get direct interactive access to driver logs from your terminal;
  you check the cluster manager's UI/logs instead.

--------------------

## Q14: Write a query to filter a dataset for rows where the region is 'North' OR the priority is 'High'.

In [28]:
result_Q14 = df.filter((col("region") == "North") | (col("priority") == "High"))
result_Q14.show()


+----------+----------+-------------+--------+----------+---------+--------+-------+------+--------+
|product_id|  new_name|     category|   price|base_price|   status|  amount|user_id|region|priority|
+----------+----------+-------------+--------+----------+---------+--------+-------+------+--------+
|     P0002|    Bottle|      Kitchen|15357.29|  13391.34|  Pending|15357.29|   1002|  East|    High|
|     P0003|     Phone|  Electronics|45308.08|  45152.67|Completed|45308.08|   1003| North|    High|
|     P0006|     Watch|  Accessories|16990.38|  15016.24|Completed|16990.38|   1006|  East|    High|
|     P0007|     Watch|  Accessories| 66935.5|  65899.55|Completed| 66935.5|   1007|  West|    High|
|     P0010|     Phone|  Electronics|81745.74|  68209.35|Cancelled|81745.74|   1010|  West|    High|
|     P0011|     Mouse|  Electronics|23005.38|  21123.57|Completed|23005.38|   1011| North|     Low|
|     P0012|Headphones|  Electronics|25793.63|  22805.24|Cancelled|25793.63|   1012| South|

--------------------

## Q15: When exploring a dataset, why is it safer to use .show(5) instead of .collect() on a multi-terabyte dataset?

- `.collect()` pulls **every single row of the entire result set** back to the Driver's JVM
  memory as a local Python/Scala list. On a multi-terabyte dataset this will either take an
  extremely long time, or (far more likely) **crash the Driver with an OutOfMemoryError**,
  since the Driver typically has a fraction of the cluster's total memory.
- `.show(5)` is itself smart: Spark doesn't compute the full result first it uses an
  internal `limit` optimization, computing only enough partitions to produce the first 5 rows
  (in many cases just scanning the first partition), and prints them directly on the Driver
  without materializing the whole dataset anywhere.
- General best practice for exploration/debugging: use `.show()`, `.limit(n).collect()`, or
  `.take(n)` to peek at data; only use full `.collect()` when you are certain the result has
  already been aggregated/filtered down to something that safely fits in Driver memory
  (e.g. after a `groupBy` that produces a handful of rows).


--------------------